# Merge Top + Secondary + Tertiary Extractions

Combines the three per-tier merged CSVs (already HTML+text merged) into one
flat dataset keyed by normalized `document_id`.

**Merge order:**
1. Load `merged_top.csv`, `merged_secondary.csv`, `merged_tertiary.csv`
2. Normalize doc IDs (remove spaces/dots/dashes, uppercase)
3. Outer-join all three on `document_id`
4. Save as `output/merged_all.csv`

*(PDF AcroForm merge is in Section 5 — run separately once ready.)*

In [10]:
import sys
from pathlib import Path
import os, csv

sys.path.insert(0, str(Path.cwd().parent))

from merge.merge_extractions import normalize_doc_id, is_valid_waiver_id, is_empty, compute_fill_rate

import pandas as pd
import numpy as np

In [3]:
# ── Configuration ──
TOP_CSV       = '../output/merged_top.csv'
SECONDARY_CSV = '../output/merged_secondary.csv'
TERTIARY_CSV  = '../output/merged_tertiary.csv'
OUTPUT_CSV    = '../output/merged_all.csv'

PDF_CSV       = '../output/pdf_acroform_extraction.csv'  # used in Section 5

## 1. Load tier CSVs

In [4]:
df_top  = pd.read_csv(TOP_CSV)
df_sec  = pd.read_csv(SECONDARY_CSV)
df_ter  = pd.read_csv(TERTIARY_CSV)

print(f'Top:       {df_top.shape}')
print(f'Secondary: {df_sec.shape}')
print(f'Tertiary:  {df_ter.shape}')

Top:       (1151, 81)
Secondary: (1151, 27)
Tertiary:  (1178, 60)


## 2. Normalize doc IDs and filter invalid rows

In [5]:
def prep(df, label):
    df = df.copy()
    df['document_id'] = df['document_id'].apply(normalize_doc_id)
    before = len(df)
    df = df[df['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
    df = df.drop_duplicates(subset=['document_id'], keep='first')
    print(f'{label}: {before} → {len(df)} rows after filtering/dedup')
    return df

df_top = prep(df_top, 'Top')
df_sec = prep(df_sec, 'Secondary')
df_ter = prep(df_ter, 'Tertiary')

Top: 1151 → 1151 rows after filtering/dedup
Secondary: 1151 → 1151 rows after filtering/dedup
Tertiary: 1178 → 1178 rows after filtering/dedup


## 3. Outer-join all three tiers on document_id

In [6]:
merged = (
    df_top
    .merge(df_sec, on='document_id', how='outer', suffixes=('', '_sec'))
    .merge(df_ter, on='document_id', how='outer', suffixes=('', '_ter'))
)

# Replace NaN strings with empty string for consistency
merged = merged.fillna('')

print(f'Merged shape: {merged.shape}')
print(f'Unique doc IDs: {merged["document_id"].nunique()}')
display(merged.head(5))

Merged shape: (1178, 166)
Unique doc IDs: 1178


,document_id,title,waiver_type,effective_date,hospital_loc,hospital_loc_limits,nursing_facility_loc,nursing_facility_loc_limits,ifc_loc,ifc_loc_limits,...,transition_plan_1,transition_plan_2,transition_plan_3,transition_plan_4,transition_plan_5,transition_plan_6,transition_plan_7,transition_plan_8,transition_plan_9,transition_plan_10
0,AK0260R0600,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/21,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AK0260R0602,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/22,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AK0260R0604,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/23,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AK0260R0607,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/24,0.0,,0.0,,1.0,,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,AK0261R0301,Older Alaskans renewal waiver,Regular Waiver,07/01/08,0.0,,1.0,,0.0,,...,,,,,,,,,,


## 4. Save merged_all.csv

In [ ]:
import os, csv

os.makedirs(os.path.dirname(OUTPUT_CSV) or '.', exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f'Saved {len(merged)} records → {OUTPUT_CSV}')

In [7]:
PDF_CSV

'../output/pdf_acroform_extraction.csv'

## 5. (Later) Merge with PDF AcroForm

Run this section once `pdf_acroform_extraction.csv` is ready.
PDF is authoritative for radio-button fields (those values are read directly
from the form state and are more reliable than text parsing).

Fields marked authoritative come from the PDF when available; all other fields
fall back to the HTML+text merged value.

In [11]:
from merge.merge_extractions import merge_two_sources

# Fields where PDF AcroForm overrides HTML+text (radio buttons only)
PDF_AUTHORITATIVE = [
    'approval_period',
    'waive_1902a',
    'waive_statewideness',
    'statecontracts_mcos1', 'statecontracts_mcos2',
    'statecontracts_mcos3', 'statecontracts_mcos4',
    'payforresidential',
    'reimburse_paidcg',
]

if os.path.exists(PDF_CSV):
    df_pdf = pd.read_csv(PDF_CSV)
    df_pdf['document_id'] = df_pdf['document_id'].apply(normalize_doc_id)
    df_pdf = df_pdf[df_pdf['document_id'].apply(is_valid_waiver_id)].reset_index(drop=True)
    print(f'PDF AcroForm: {df_pdf.shape}')

    merged_final = merge_two_sources(
        merged, df_pdf,
        source_a_name='top+sec+ter',
        source_b_name='pdf_acroform',
        authoritative_fields=PDF_AUTHORITATIVE,
        authoritative_source='b',
    )

    FINAL_CSV = OUTPUT_CSV.replace('merged_all', 'merged_all_with_pdf')
    merged_final.to_csv(FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)
    print(f'Saved {len(merged_final)} records → {FINAL_CSV}')
else:
    print(f'PDF CSV not found at {PDF_CSV} — skipping.')

PDF AcroForm: (779, 20)
Source top+sec+ter: 1178 unique IDs
Source pdf_acroform: 779 unique IDs
  Only in top+sec+ter: 442
  Only in pdf_acroform: 43
  Overlap: 736

Merged: 1221 rows, 185 columns
Saved 1221 records → ../output/merged_all_with_pdf.csv


In [12]:
import os, csv

PDF_FINAL_CSV = "../output/merged_all_with_pdf.csv"

df_pdf = pd.read_csv(PDF_CSV)
df_pdf["document_id"] = df_pdf["document_id"].apply(normalize_doc_id)
df_pdf = df_pdf[df_pdf["document_id"].apply(is_valid_waiver_id)].reset_index(drop=True)
df_pdf = df_pdf.drop_duplicates(subset=["document_id"], keep="first")
print(f"PDF AcroForm: {df_pdf.shape}")

merged_with_pdf = merged.merge(df_pdf, on="document_id", how="left")
merged_with_pdf = merged_with_pdf.fillna("")

# ── Column order ──────────────────────────────────────────────────────────────
FINAL_COLUMN_ORDER = [
    # Key
    "document_id",
    # Waiver overview
    "title",
    "waiver_type",
    "effective_date",
    "approval_period",
    # Level of care
    "hospital_loc",
    "hospital_loc_limits",
    "nursing_facility_loc",
    "nursing_facility_loc_limits",
    "ifc_loc",
    "ifc_loc_limits",
    "local_eval",
    "local_eval_instrument",
    "reeval_sched",
    # Concurrent waivers
    "concurrent_1915a",
    "concurrent_1915b",
    "concurrent_1932a",
    "concurrent_1915i",
    "concurrent_1915j",
    "concurrent_1115",
    # Eligibility — geography & groups
    "dual_elg",
    "waive_geographic_limits",
    "waive_geographic_lipd",
    "aged_group",
    "aged_group_min",
    "aged_group_max",
    "physicaldis_group",
    "physicaldis_group_min",
    "physicaldis_group_max",
    "otherdis_group",
    "otherdis_group_min",
    "otherdis_group_max",
    "braininjury_group",
    "braininjury_group_min",
    "braininjury_group_max",
    "hivaids_group",
    "hivaids_group_min",
    "hivaids_group_max",
    "medicallyfrail_group",
    "medicallyfrail_group_min",
    "medicallyfrail_group_max",
    "techdep_group",
    "techdep_group_min",
    "techdep_group_max",
    "autism_group",
    "autism_group_min",
    "autism_group_max",
    "dd_group",
    "dd_group_min",
    "dd_group_max",
    "id_group",
    "id_group_min",
    "id_group_max",
    "mi_group",
    "mi_group_min",
    "mi_group_max",
    "sed_group",
    "sed_group_min",
    "sed_group_max",
    # Eligibility — criteria
    "eligibility_1",
    "eligibility_2",
    "eligibility_3",
    "eligibility_4",
    "eligibility_5",
    "eligibility_5_percent",
    "eligibility_5_100",
    "eligibility_6",
    "eligibility_7",
    "eligibility_8",
    "eligibility_9",
    "eligibility_10",
    "eligibility_11",
    "eligibility_12",
    "spousal_impov_a",
    "spousal_impov_bc",
    # Cost / enrollment
    "cost_limit_pcntaboveinstit",
    "costlimit",
    "numberofbenes_year1",
    "numberofbenes_year2",
    "numberofbenes_year3",
    "numberofbenes_year4",
    "numberofbenes_year5",
    "max_numberofbenes_year1",
    "max_numberofbenes_year2",
    "max_numberofbenes_year3",
    "max_numberofbenes_year4",
    "max_numberofbenes_year5",
    "numberbenes_limited",
    "phaseinoutschedule",
    "entrantselection",
    "specialHCBS",
    "enhanced_payments_yes",
    "statecontracts_mcos",
    # Self-direction (Appendix E)
    "selfdirection_yes",
    "selfdirection_description",
    "sd_authority",
    "sd_election",
    "sd_livarrngmt_1",
    "sd_livarrngmt_2",
    "sd_livarrngmt_3",
    "sd_service_1",
    "sd_service_1_ea",
    "sd_service_1_ba",
    "sd_fms_gov",
    "sd_fms_pe",
    "scope_fms_1",
    "scope_fms_2",
    "scope_fms_3",
    "scope_fms_4",
    "sd_numenrollees_ea1",
    "sd_numenrollees_ea2",
    "sd_numenrollees_ea3",
    "sd_numenrollees_ea4",
    "sd_numenrollees_ea5",
    "sd_numenrollees_ba1",
    "sd_numenrollees_ba2",
    "sd_numenrollees_ba3",
    "sd_numenrollees_ba4",
    "sd_numenrollees_ba5",
    "sd_coemployer",
    "sd_commonlaw",
    "provider_rate_methods",
    "payforresidential",
    "reimburse_paidcg",
    # Waiver services (Tertiary)
    "ma_1",
    "ma_2",
    "ma_3",
    "ma_4",
    "ma_5",
    "ma_6",
    "ma_7",
    "ma_8",
    "ma_9",
    "ma_10",
    "ma_11",
    "ma_12",
    "osa_1",
    "osa_2",
    "osa_3",
    "osa_4",
    "osa_5",
    "osa_6",
    "osa_7",
    "osa_8",
    "osa_9",
    "osa_10",
    "osa_11",
    "osa_12",
    "ce_1",
    "ce_2",
    "ce_3",
    "ce_4",
    "ce_5",
    "ce_6",
    "ce_7",
    "ce_8",
    "ce_9",
    "ce_10",
    "ce_11",
    "ce_12",
    "inse_1",
    "inse_2",
    "inse_3",
    "inse_4",
    "inse_5",
    "inse_6",
    "inse_7",
    "inse_8",
    "inse_9",
    "inse_10",
    "inse_11",
    "inse_12",
    # Descriptions / Transition
    "waiver_description",
    "transitionplan_1",
    "transitionplan_2",
    "transitionplan_3",
    "transitionplan_4",
    "transitionplan_5",
    "transitionplan_6",
    "transitionplan_7",
    "transitionplan_8",
    "transitionplan_9",
    "transitionplan_10",
]

# Keep only columns that actually exist (guards against future schema changes)
final_cols = [c for c in FINAL_COLUMN_ORDER if c in merged_with_pdf.columns]
missing = [c for c in FINAL_COLUMN_ORDER if c not in merged_with_pdf.columns]
if missing:
    print(f"Warning — columns in order list but not in data: {missing}")

merged_with_pdf = merged_with_pdf[final_cols]

print(f"Final shape: {merged_with_pdf.shape}")
merged_with_pdf.to_csv(PDF_FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved {len(merged_with_pdf)} records → {PDF_FINAL_CSV}")

display(merged_with_pdf.head(3))

PDF AcroForm: (779, 20)
Final shape: (1178, 183)
Saved 1178 records → ../output/merged_all_with_pdf.csv


,document_id,title,waiver_type,effective_date,approval_period,hospital_loc,hospital_loc_limits,nursing_facility_loc,nursing_facility_loc_limits,ifc_loc,...,transition_plan_1,transition_plan_2,transition_plan_3,transition_plan_4,transition_plan_5,transition_plan_6,transition_plan_7,transition_plan_8,transition_plan_9,transition_plan_10
0,AK0260R0600,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/21,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,AK0260R0602,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/22,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AK0260R0604,People with Intellectual and Developmental Dis...,Regular Waiver,07/01/23,5 years,0.0,,0.0,,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
# merge the misc Vs merged all(text+html+acroform)
# Set KEEP_MISC_ONLY = True to also append misc-only doc IDs

KEEP_MISC_ONLY = True  # ← change to True to include misc-only rows

import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

def clean_id(v):
    if pd.isna(v): return ''
    return str(v).strip()

base = pd.read_csv('../output/merged_all_with_pdf.csv')
misc = pd.read_csv('../output/misc_extraction2.csv')

base['document_id'] = base['document_id'].apply(clean_id)
misc['document_id'] = misc['document_id'].apply(clean_id)

ids_base = set(base['document_id'])

# Fill empty cells for matching doc IDs
misc_lookup = misc.set_index('document_id').to_dict('index')
shared_cols = [c for c in misc.columns if c in base.columns and c != 'document_id']

filled = 0
for idx, row in base.iterrows():
    doc_id = row['document_id']
    misc_row = misc_lookup.get(doc_id, {})
    for col in shared_cols:
        if is_empty(row[col]) and not is_empty(misc_row.get(col)):
            base.at[idx, col] = misc_row[col]
            filled += 1

if KEEP_MISC_ONLY:
    only_misc = misc[~misc['document_id'].isin(ids_base)].copy()
    print(f'Appending {len(only_misc)} misc-only rows')
    base = pd.concat([base, only_misc], ignore_index=True)
else:
    print('misc-only rows dropped (KEEP_MISC_ONLY=False)')

base = base.sort_values('document_id').reset_index(drop=True)
base.to_csv('../output/merged_all_with_pdf_updated.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Cells filled from misc: {filled}')
print(f'Final shape: {base.shape}')


/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  base.at[idx, col] = misc_row[col]
/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  base.at[idx, col] = misc_row[col]
/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/1033282454.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'No section' has dtype incompatible with float64, please explicitly cast to a compatible dt

Appending 39 misc-only rows
Cells filled from misc: 17429
Final shape: (1217, 186)


In [27]:
# applying athe clean text pipeline ( reomve url, page numbers, etc) to the merged_all_with_pdf_updated.csv output from above, and save back to same file (overwriting it)
import pandas as pd, csv, sys
import importlib, sys

# Remove cached module so changes are picked up without restarting kernel
if 'merge.merge_extractions' in sys.modules:
    importlib.reload(sys.modules['merge.merge_extractions'])
from merge.merge_extractions import clean_text_value


df = pd.read_csv('../output/merged_all_with_pdf_updated.csv')
str_cols = [c for c in df.columns if df[c].dtype == object]
for col in str_cols:
    df[col] = df[col].apply(clean_text_value)

df.to_csv('../output/merged_all_with_pdf_updated.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Cleaned {len(str_cols)} string columns. Shape: {df.shape}')

Cleaned 60 string columns. Shape: (1217, 186)


In [28]:
# drop non values 

import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

df = pd.read_csv('../output/merged_all_with_pdf_updated.csv')

# Drop rows with no document_id
before_id = len(df)
df = df[df['document_id'].notna() & (df['document_id'].str.strip() != '')]
print(f'Dropped no doc_id: {before_id - len(df)}')

# Filter rows with >= 80% missing
data_cols = [c for c in df.columns if c != 'document_id']
df['missing_pct'] = df[data_cols].apply(
    lambda row: row.apply(is_empty).sum() / len(data_cols) * 100, axis=1
).round(1)

before = len(df)
df_clean = df[df['missing_pct'] < 80].drop(columns='missing_pct')
dropped  = before - len(df_clean)

df_clean.to_csv('../output/merged_all_with_pdf_drop_nan.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Before: {before} | Dropped (>=80% missing): {dropped} | Remaining: {len(df_clean)}')


Dropped no doc_id: 0
Before: 1217 | Dropped (>=80% missing): 61 | Remaining: 1156


In [29]:
import pandas as pd, csv, sys
sys.path.insert(0, '..')
from merge.merge_extractions import is_empty

df = pd.read_csv('../output/merged_all_with_pdf_drop_nan.csv')
before = len(df)

# 1. Drop rows with null/empty document_id OR null title
df = df[
    df['document_id'].apply(lambda v: not is_empty(v)) &
    df['title'].apply(lambda v: not is_empty(v))
].reset_index(drop=True)
print(f'Dropped: {before - len(df)} | Remaining: {len(df)}')

# 2. Standardize waiver_type
valid_types = {'Model Waiver', 'Regular Waiver'}
def fix_waiver_type(v):
    if is_empty(v): return 'Regular Waiver'
    s = str(v).strip()
    return s if s in valid_types else 'Regular Waiver'

changed_wt = (df['waiver_type'] != df['waiver_type'].apply(fix_waiver_type)).sum()
df['waiver_type'] = df['waiver_type'].apply(fix_waiver_type)
print(f'waiver_type corrected: {changed_wt}')

# 3. Fill empty approval_period with '5 years'
filled_ap = df['approval_period'].apply(is_empty).sum()
df['approval_period'] = df['approval_period'].apply(
    lambda v: '5 years' if is_empty(v) else v
)
print(f'approval_period filled with "5 years": {filled_ap}')

df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Final shape: {df.shape}')


Dropped: 16 | Remaining: 1140
waiver_type corrected: 17
approval_period filled with "5 years": 411
Final shape: (1140, 186)


In [ ]:
## file based clean up

import re

# Fix min_numservices where section text was captured instead of just the number
# re.match(r'^(\d+)\s*ii\.', x) extracts the leading 1 and returns it. The last case (ii. Frequency... with no leading number) gets blanked. 
def clean_min_numservices(x):
    x = str(x).strip()
    if not x or x in ('', 'nan', 'None'):
        return ''
    # "1ii. Frequency of services..." → "1"
    m = re.match(r'^(\d+)\s*ii\.', x)
    if m:
        return m.group(1)
    # Pure label text with no leading number → blank
    if 'Frequency of services' in x:
        return ''
    return x

if 'min_numservices' in df.columns:
    df['min_numservices'] = df['min_numservices'].apply(clean_min_numservices)
    print('min_numservices cleaned')

# Flag entrantselection rows that captured wrong section text
bad_entrant_patterns = [
    '1. Request Information',
    '3. Nature of the Amendment',
    'Component(s) of the Approved Waiver',
]

flags = []
for idx, row in df.iterrows():
    val = str(row.get('entrantselection', '')).strip()
    if val and val not in ('', 'nan', 'None'):
        if any(p in val for p in bad_entrant_patterns):
            flags.append({
                'document_id': row['document_id'],
                'issue': 'Possible wrong section captured in entrantselection',
                'value': val[:300],
            })

flags_df = pd.DataFrame(flags)
print(f'entrantselection flags: {len(flags_df)}')
display(flags_df)


min_numservices cleaned
entrantselection flags: 7


,document_id,issue,value
0,IL0350R0501,Possible wrong section captured in entrantsele...,3. Nature of the Amendment A. Component(s) of ...
1,LA0121R0504,Possible wrong section captured in entrantsele...,"- Update the ""Adult Day Health Care"" (ADHC) se..."
2,LA0866R0004,Possible wrong section captured in entrantsele...,- Add Housing Stabilization and Housing Transi...
3,LA0866R0200,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
4,OH0337R0500,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
5,TX0221R0700,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...
6,UT0292R0600,Possible wrong section captured in entrantsele...,1. Request Information (1 of 3) A. The State o...


In [33]:
bad_ids = {
    'IL0350R0501', 'LA0121R0504', 'LA0866R0004',
    'LA0866R0200', 'OH0337R0500', 'TX0221R0700', 'UT0292R0600'
}

mask = df['document_id'].isin(bad_ids)
df.loc[mask, 'entrantselection'] = ''
print(f'Blanked entrantselection for {mask.sum()} rows')


Blanked entrantselection for 7 rows


In [37]:
import re

# 1. Fix § encoding artifact in statecontracts_mcos
if 'statecontracts_mcos' in df.columns:
    df['statecontracts_mcos'] = df['statecontracts_mcos'].str.replace(
        r'\?1115', '§1115', regex=True
    ).str.replace(
        r'\?1915', '§1915', regex=True
    )

# 2. Fix workers' compensation encoding artifact
for col in df.columns:
    df[col] = df[col].astype(str).str.replace(
        r"workersâ\s+compensation", "workers' compensation", regex=True, flags=re.IGNORECASE
    )

# 3. Convert 0.0 / 1.0 → 0 / 1 across all columns
def clean_decimal_int(val):
    if isinstance(val, float) and val == int(val):
        return str(int(val))
    if isinstance(val, str):
        try:
            f = float(val)
            if f == int(f):
                return str(int(f))
        except (ValueError, OverflowError):
            pass
    return val
# Replace all nan strings with empty string
df = df.fillna('').astype(str).replace({'nan': '', 'None': '', '<NA>': ''})

df = df.applymap(clean_decimal_int)

print('Done. Shape:', df.shape)


/var/folders/8s/tf002srj1_309fmkfswmhbv00000gn/T/ipykernel_17535/525681540.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_decimal_int)


Done. Shape: (1140, 186)


In [38]:
import csv

df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
print(f'Saved. Shape: {df.shape}')


Saved. Shape: (1140, 186)


In [39]:
##### Final saving and converting to dta file

# df.to_csv('../output/merged_all_with_pdf_drop_nan_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)
df.to_stata('../output/1915c-waiver-level-v2.dta', write_index=False, version=118)
df.to_csv('../output/1915c-waiver-level-v2.csv', index=False, quoting=csv.QUOTE_ALL)
print('Saved.')


Saved.
